# DDVC YouCook2: Action-aware, GT-supervised learned masks

New branch: `exp/action-aware-grounded-mask`. Random-initialized GPT-2; original DDVC retrieval pipeline retained.

Adds: predicted-context attention pruning, balanced mask BCE on final Hungarian-matched GT only, caption-query-conditioned mask MLP, bounded class-confidence gate (0.1–0.85), and 10% context on EACH side with minimum 4 frames. Corrected query-contrastive temperature/self-negative/warmup recipe is enabled for both improved and global controls. It changes training relative to historical R15: compare with a new corrected global control before attributing gains to masking.

**Default: subset 30%, 15 epochs, seed 777, batch 1, 50 queries, one T4. Full also defaults to 15 epochs.** `RUN_MODE='smoke'` selects 1% and 2 epochs. GPU and Internet required; no Input dataset required. Set `MASK_MODE='global'` or `'hard_local'` for controls; keep every other setting identical. Each improvement has its own config switch.

Atomic format-2 checkpoints include optimizer/scheduler/RNG/config and pending-validation phase. Resume interrupted validation without retraining its completed epoch. **Do not resume historical R15 checkpoints with this new architecture**: use a fresh run. Save model-last/model-best separately from analysis ZIP. CPU preflight is not a GPU/metric guarantee.


In [ ]:
from pathlib import Path
import os, sys, json, random, subprocess, time, collections
import torch

DATA_ROOT = Path('/kaggle/working/youcook2_ddvc_data')
RUN_MODE = 'subset'  # 'smoke' = 1%, 'subset' = 30%, 'full' = all feature-matched videos
SMOKE_FRACTION = 0.01
SUBSET_FRACTION = 0.30
if RUN_MODE not in ('smoke', 'subset', 'full'):
    raise ValueError("RUN_MODE must be 'smoke', 'subset', or 'full'")
ACTIVE_FRACTION = SMOKE_FRACTION if RUN_MODE == 'smoke' else (SUBSET_FRACTION if RUN_MODE == 'subset' else 1.0)
EPOCHS = 2 if RUN_MODE == 'smoke' else 15
SEED = 777
MASK_MODE = 'learned_gated'  # global / hard_local / learned_gated
MASK_BINS = 64
MASK_ENCODING_DIM = 128
MASK_LOSS_COEF = 1.0
# Each switch can be ablated. Start a fresh run when architecture/recipe changes.
MASK_TARGET = 'matched_gt'  # predicted / matched_gt (training only)
QUERY_CONDITIONING = True
PRUNE_THRESHOLD = 0.05     # -1 disables pruning; predicted context is always retained
GATE_MIN, GATE_MAX = 0.1, 0.85
MASK_CONTEXT_RATIO, MASK_MIN_FRAMES = 0.10, 4  # margin on EACH side
BALANCED_MASK_BCE = True
CORRECTED_QUERY_CONTRASTIVE = True  # keep enabled for corrected global control too
LOCAL_CONTEXT_RATIO = 0.10  # only used by hard_local control
LOCAL_MIN_FRAMES = 4
if MASK_MODE not in ('global','hard_local','learned_gated'):
    raise ValueError('MASK_MODE must be global, hard_local or learned_gated')
RESUME_RUN_ID = ''  # leave blank for a fresh run; set to this run ID to resume model-last.pth
RUN_ID = RESUME_RUN_ID or f'yc2_{RUN_MODE}_{MASK_MODE}_action_mask'
REPO = Path('/kaggle/working/DDVC_action_aware_mask')
SAVE_DIR = 'save_action_aware_mask'
LOG_PATH = Path(f'/kaggle/working/ddvc_{MASK_MODE}_action_mask_train.log')
SOURCE_COMMIT = '80c173b8e66314439ff83de30bfffb8b3af7877b'
assert torch.cuda.is_available(), 'Enable a GPU in Kaggle Settings.'
print('Mode:', RUN_MODE, '| fraction per split:', f'{ACTIVE_FRACTION:.0%}', '| epochs:', EPOCHS)
print('GPU:', torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/2**30, 1), 'GiB')
print('Data will be downloaded into', DATA_ROOT)

print('Mask mode:',MASK_MODE,'| mask BCE coefficient:',MASK_LOSS_COEF)


## Clone repo, cài runtime, build CUDA op

Bản upstream đọc GPT-2 tokenizer từ đường dẫn riêng của tác giả; notebook chuyển sang tên model public. Extension Deformable Attention cần build cho compute capability 7.5 của T4.

In [ ]:
REPO_URL = 'https://github.com/thloc1212/DDVC.git'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'exp/action-aware-grounded-mask', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', '--depth', '1', REPO_URL, SOURCE_COMMIT], check=True)
# Clear Kaggle-only compatibility edits from an earlier interrupted run before switching revisions.
subprocess.run(['git', '-C', str(REPO), 'reset', '--hard'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', SOURCE_COMMIT], check=True)
actual_commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_commit == SOURCE_COMMIT, (actual_commit, SOURCE_COMMIT)
print('DDVC source commit:', actual_commit)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'PyYAML', 'tensorboardX', 'colorlog', 'ftfy', 'regex', 'gdown', 'pycocotools', 'setuptools<81', 'wheel'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', '--no-deps', 'git+https://github.com/openai/CLIP.git@dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1'], check=True)
# Patch deprecated PyTorch APIs used by this legacy CUDA extension.
ops_dir = REPO/'ddvc'/'ops'
cuda_src = ops_dir/'src'/'cuda'/'ms_deform_attn_cuda.cu'
cuda_text = cuda_src.read_text()
legacy, fixed = 'AT_DISPATCH_FLOATING_TYPES(value.type(),', 'AT_DISPATCH_FLOATING_TYPES(value.scalar_type(),'
if legacy in cuda_text:
    cuda_text = cuda_text.replace(legacy, fixed)
elif fixed not in cuda_text:
    raise RuntimeError(f'Could not apply PyTorch compatibility patch to {cuda_src}')
cuda_text = cuda_text.replace('.data<', '.data_ptr<')
cuda_src.write_text(cuda_text)

# Discard partial objects from an interrupted compile and avoid Ninja launching
# more compiler processes than Kaggle's small CPU/RAM allocation can sustain.
import shutil
shutil.rmtree(ops_dir/'build', ignore_errors=True)
env = os.environ.copy()
env.update({'TORCH_CUDA_ARCH_LIST':'7.5', 'MAX_JOBS':'2'})
print('Building the CUDA extension with MAX_JOBS=2 ...', flush=True)
subprocess.run([sys.executable, 'setup.py', 'build', 'install'], cwd=ops_dir, env=env, check=True, timeout=1200)
print('CUDA extension build and install completed.', flush=True)


## Tải annotation chính thức và CLIP features của DDVC

Không cần chọn hay gắn dataset trong Kaggle Input. Bật Internet để notebook tải annotation YouCook2 chính thức và file CLIP ViT-L/14 mà repo DDVC cung cấp. Sau đó notebook đối chiếu ID video, in số liệu độ phủ, và chỉ loại video không có feature tương ứng.


In [ ]:
import json, tarfile, requests
from pathlib import Path

DATA_ROOT.mkdir(parents=True, exist_ok=True)


def download_file(url, destination):
    destination=Path(destination)
    if destination.exists() and destination.stat().st_size > 0:
        return destination
    print('Downloading:', url)
    partial=destination.with_name(destination.name+'.part')
    with requests.get(url, stream=True, timeout=(30, 180)) as response:
        response.raise_for_status()
        with partial.open('wb') as f:
            for chunk in response.iter_content(chunk_size=8*1024*1024):
                if chunk:
                    f.write(chunk)
    partial.replace(destination)
    return destination


def convert_youcook2(raw):
    database=raw.get('database', raw)
    splits={'training':{}, 'validation':{}}
    for video_id, item in database.items():
        subset=item.get('subset')
        if subset not in splits:
            continue
        annotations=[a for a in item.get('annotations', [])
                     if a.get('sentence') and isinstance(a.get('segment'), (list, tuple)) and len(a['segment']) == 2]
        if not annotations:
            continue
        splits[subset][video_id]={
            'duration': float(item['duration']),
            'sentences': [a['sentence'] for a in annotations],
            'timestamps': [[float(a['segment'][0]), float(a['segment'][1])] for a in annotations],
        }
    return splits['training'], splits['validation']


# Official YouCook2 combined train/validation annotations.
try:
    ann_tar=download_file(
        'https://youcook2.eecs.umich.edu/static/YouCookII/youcookii_annotations_trainval.tar.gz',
        DATA_ROOT/'youcookii_annotations_trainval.tar.gz')
    with tarfile.open(ann_tar, 'r:gz') as archive:
        members=[m for m in archive.getmembers() if Path(m.name).name == 'youcookii_annotations_trainval.json' and m.isfile()]
        if len(members) != 1:
            raise FileNotFoundError('Official annotation archive did not contain exactly one youcookii_annotations_trainval.json')
        with archive.extractfile(members[0]) as f:
            raw_annotations=json.load(f)
except requests.exceptions.SSLError:
    # Kaggle mirror of the same official annotation file, used only if the UMich TLS chain fails.
    print('Official host SSL verification failed; using the Kaggle mirror of the official annotation file.')
    ann_json=download_file(
        'https://www.kaggle.com/api/v1/datasets/download/ravishkumarcs23m055/annotation-youcook2/youcookii_annotations_trainval.json',
        DATA_ROOT/'youcookii_annotations_trainval.json')
    with ann_json.open(encoding='utf-8') as f:
        raw_annotations=json.load(f)

train_all,val_all=convert_youcook2(raw_annotations)
del raw_annotations
assert train_all and val_all, 'Official YouCook2 train/validation annotations did not load.'
print('Official annotations:',len(train_all),'train videos,',len(val_all),'validation videos')

# DDVC's pre-extracted CLIP ViT-L/14 features (linked from its official repository README).
feature_src=DATA_ROOT/'clipvitl14.pth'
if not feature_src.exists() or feature_src.stat().st_size < 100*1024*1024:
    if feature_src.exists(): feature_src.unlink()
    import gdown
    downloaded=gdown.download(
        id='17H_lxSKFve57kHpkAD7pcYS7ijHIy4-M',
        output=str(feature_src), quiet=False)
    if downloaded is None or not feature_src.exists() or feature_src.stat().st_size < 100*1024*1024:
        raise RuntimeError('CLIP feature download failed or is incomplete. Enable Kaggle Internet and retry; expected the DDVC clipvitl14.pth file.')
print('DDVC CLIP features:',feature_src,round(feature_src.stat().st_size/1024**2),'MiB')

for split,data in [('train',train_all),('validation',val_all)]:
    assert isinstance(data,dict) and data, f'{split} annotation split is empty'
    assert {'duration','sentences','timestamps'} <= set(next(iter(data.values()))), f'{split}.json has an unexpected format'
try:
    feature_map=torch.load(feature_src,map_location='cpu',weights_only=False)
except TypeError:
    feature_map=torch.load(feature_src,map_location='cpu')
if 'features' in feature_map and isinstance(feature_map['features'],dict):
    feature_map=feature_map['features']
assert isinstance(feature_map,dict), f'Expected a video-ID to feature mapping in {feature_src}'
feature_ids=set(feature_map)


def choose(data,n,seed,split_name):
    valid_ids=[k for k,v in data.items() if k in feature_ids and v.get('sentences') and len(v['sentences'])==len(v['timestamps'])]
    missing_ids=set(data)-feature_ids
    random.Random(seed).shuffle(valid_ids)
    if not valid_ids:
        raise ValueError(f'No {split_name} videos match the downloaded CLIP feature IDs.')
    selected_ids=valid_ids if n is None else valid_ids[:min(n,len(valid_ids))]
    selected={k:data[k] for k in selected_ids}
    events_all=sum(len(v['sentences']) for v in data.values())
    events_selected=sum(len(v['sentences']) for v in selected.values())
    print(f'{split_name}: {len(data):,} official annotated videos / {len(valid_ids):,} with features / {len(selected):,} used; {events_all:,} annotations / {events_selected:,} used')
    print(f'  Missing feature for {len(missing_ids):,} annotated videos; examples: {sorted(missing_ids)[:8]}')
    print(f'  Feature-matched but omitted by subset limit: {len(set(valid_ids)-set(selected_ids)):,}')
    return selected

train_available=sum(k in feature_ids for k in train_all)
val_available=sum(k in feature_ids for k in val_all)
TRAIN_VIDEOS=None if RUN_MODE=='full' else max(1, round(train_available*ACTIVE_FRACTION))
VAL_VIDEOS=None if RUN_MODE=='full' else max(1, round(val_available*ACTIVE_FRACTION))
train=choose(train_all,TRAIN_VIDEOS,SEED,'train')
val=choose(val_all,VAL_VIDEOS,SEED+1,'validation')
assert not (set(train)&set(val)), 'Train/validation video IDs overlap.'

REPO_DATA=REPO/'data'/'yc2'
ANNO=REPO_DATA/'captiondata'
ANNO.mkdir(parents=True,exist_ok=True)
train_json,val_json=ANNO/'train.json',ANNO/'val.json'
train_json.write_text(json.dumps(train),encoding='utf-8')
val_json.write_text(json.dumps(val),encoding='utf-8')

# Upstream paragraph evaluator expects video_id -> concatenated reference captions.
para_dir=ANNO/'para'
para_dir.mkdir(parents=True,exist_ok=True)
para_json=para_dir/'para_yc2_val.json'
para_refs={}
for vid,record in val.items():
    ordered=sorted(zip(record['timestamps'],record['sentences']),key=lambda pair:float(pair[0][0]))
    para_refs[vid]=' '.join(sentence.strip().rstrip('.')+'.' for _,sentence in ordered)
para_json.write_text(json.dumps(para_refs),encoding='utf-8')
print('Wrote full matched validation references:',para_json,'videos:',len(para_refs))

small={k:torch.as_tensor(feature_map[k]).detach().cpu().float() for k in set(train)|set(val)}
features_path=REPO_DATA/'features'/f'clipvitl14_{RUN_MODE}.pth'
features_path.parent.mkdir(parents=True,exist_ok=True)
torch.save(small,features_path)
del feature_map,small
print('Prepared selected data:',len(train),'train,',len(val),'validation videos; feature file:',features_path)


## CLIP retrieval memory for the selected training subset

The original DDVC retrieval mechanism is retained. The memory bank is built only from the selected training captions; validation captions are not included.


In [ ]:
def words(s):
    for ch in [',',':','!','_',';','-','.','?','/','"','\\n','\\']:
        s=s.lower().replace(ch,' ')
    return [x for x in s.split() if x]
counts=collections.Counter(w for v in train.values() for s in v['sentences'] for w in words(s))
vocab=[w for w,n in counts.items() if n>=2]+['<bos>','<eos>','UNK']
word_to_ix={w:i+1 for i,w in enumerate(vocab)}
vocab_path=REPO_DATA/'vocabulary_youcook2.json'
vocab_path.write_text(json.dumps({'word_to_ix':word_to_ix,'ix_to_word':{str(i):w for w,i in word_to_ix.items()}}))
import clip
sentences=[]; video_ids=[]
for vid,record in train.items():
    for sent in record['sentences']:
        sentences.append(sent.strip()); video_ids.append(vid)
model_clip,_=clip.load('ViT-L/14',device='cuda:0',jit=False); model_clip.eval()
encoded=[]
with torch.inference_mode():
    for j in range(0,len(sentences),64):
        tokens=clip.tokenize(sentences[j:j+64],truncate=True).to('cuda:0')
        encoded.append(model_clip.encode_text(tokens).float().cpu())
emb=torch.cat(encoded).numpy()
assert emb.shape==(len(sentences),768), emb.shape
for folder in ['clip','clip_arr_split']:
    out=REPO/'bank'/'yc2'/folder; out.mkdir(parents=True,exist_ok=True)
    import numpy as np
    np.save(out/'yc2_scene_sentences.npy',np.asarray(sentences,dtype=str))
    np.save(out/'yc2_clip_token_embeds.npy',emb)
    np.save(out/'yc2_scene_videoID.npy',np.asarray(video_ids,dtype=str))
print('CLIP memory entries:',len(sentences),emb.shape)
del model_clip,encoded,emb; torch.cuda.empty_cache()

## Runtime compatibility and offline CPU preflight

Apply the established Kaggle fixes, then validate the actual caption path with a tiny randomly initialized GPT-2: mask gradients, predicted timestamp gradients, frozen CLIP features, padding, query pairing, two-layer matched caption training and beam/cached generation. No model/data downloads are needed for these CPU tests.


In [ ]:
# Match the vendored Transformers dependency range; Kaggle currently ships huggingface-hub 1.x.
subprocess.run([sys.executable,'-m','pip','install','-q','huggingface-hub<1.0'],check=True)

# Restore upstream single-GPU files if this cell was run before with the old 2-GPU adapter.
restore_files=[
    'train.py',
    'eval_utils_clip.py',
    'data/video_dataset.py',
    'ddvc/CaptioningHead/gpt2.py',
    'ddvc/ddvc_ret_encdec_clip_capQ.py',
    'transformers/dependency_versions_table.py',
]
subprocess.run(['git','-C',str(REPO),'checkout','--',*restore_files],check=True)
adapter_path=REPO/'kaggle_parallel.py'
if adapter_path.exists(): adapter_path.unlink()

# Replace the author's private tokenizer cache path with the public Hugging Face model id.
for rel in ['data/video_dataset.py','ddvc/CaptioningHead/gpt2.py']:
    p=REPO/rel
    s=p.read_text()
    s=s.replace("GPT2Tokenizer.from_pretrained('/DATA_DISK/plm_cache/gpt2')", "GPT2Tokenizer.from_pretrained('gpt2')")
    p.write_text(s)

# The vendored Transformers package predates Kaggle's Python 3.13 tokenizers wheel.
# Its tokenizer API is available, but the old upper-bound check rejects 0.23.x at import time.
dep=REPO/'transformers'/'dependency_versions_table.py'
s=dep.read_text()
old_req='"tokenizers": "tokenizers>=0.19,<0.20",'
new_req='"tokenizers": "tokenizers>=0.19,<0.24",'
if old_req in s:
    dep.write_text(s.replace(old_req,new_req))
elif new_req not in s:
    raise RuntimeError('Could not find vendored Transformers tokenizers version constraint')
# YouCook2 setup has no ActivityNet grounding GT file. Keep training/evaluation intact,
# but skip this unrelated grounding-only metric during validation.
eval_path=REPO/'eval_utils_clip.py'
eval_src=eval_path.read_text()
old_grounding="""    scores_g = eval_metrics_grounding(dvc_json_path + '.grounding.json', gt_filename=opt.eval_gt_file_for_grounding)
    aux_scores_g = eval_metrics_grounding(dvc_json_path + '_aux.grounding.json', gt_filename=opt.eval_gt_file_for_grounding)
"""
new_grounding="""    if getattr(opt, 'eval_enable_grounding', True):
        scores_g = eval_metrics_grounding(dvc_json_path + '.grounding.json', gt_filename=opt.eval_gt_file_for_grounding)
        aux_scores_g = eval_metrics_grounding(dvc_json_path + '_aux.grounding.json', gt_filename=opt.eval_gt_file_for_grounding)
    else:
        scores_g, aux_scores_g = {}, {}
"""
if old_grounding in eval_src:
    eval_src=eval_src.replace(old_grounding,new_grounding,1)
elif new_grounding not in eval_src:
    raise RuntimeError('Could not locate grounding metric calls in eval_utils_clip.py')
eval_path.write_text(eval_src)
print('Patched YouCook2 validation to skip ActivityNet-only grounding metrics.')

print('Restored upstream single-GPU training; applied Kaggle tokenizers compatibility check and public GPT-2 tokenizer.')

# Stop before training if the pinned source or local-mask wiring is missing.
assert 'raw_dvc_json_path' in (REPO/'eval_utils_clip.py').read_text()
assert 'event_visual_mask' in (REPO/'ddvc'/'CaptioningHead'/'gpt2.py').read_text()
subprocess.run([sys.executable, '-u', 'smoke_test_local_visual.py'], cwd=REPO, check=True)
subprocess.run([sys.executable,'-u','smoke_test_proposal_mask.py'],cwd=REPO,check=True)
print('PASS: learned-mask and baseline caption CPU preflights.')

subprocess.run([sys.executable,'-u','smoke_test_training_state.py'],cwd=REPO,check=True)


## Configure DDVC + proposal mask

Only the mask path/objective is new. Existing DDVC task losses, matcher, query count, data selection and caption-cost chunking are retained. The reference repository is recorded in the manifest; it is not installed as an old Python dependency.


In [ ]:
import yaml
opts=json.loads((REPO/'opts.json').read_text())
import hashlib
data_signature = hashlib.sha256(train_json.read_bytes()+val_json.read_bytes()+vocab_path.read_bytes()).hexdigest()
opts.update({
 'cfg_path':'','target_domain':'yc2','dataset':'yc2','bank_type':['yc2'],
 'train_caption_file':str(train_json.relative_to(REPO)),
 'val_caption_file':str(val_json.relative_to(REPO)),
 'gt_file_for_eval':[str(val_json.relative_to(REPO))],
 'gt_file_for_auc':[str(val_json.relative_to(REPO))],
 'gt_file_for_para_eval':[str(para_json.relative_to(REPO))],
 'visual_feature_folder':[str(features_path.relative_to(REPO))],
 'visual_feature_type':['clip'],'feature_dim':768,'frame_embedding_num':200,
 'dict_file':str(vocab_path.relative_to(REPO)),'vocab_size':len(word_to_ix),
 'bank_path':'bank','bank_folder':'clip','able_ret':True,
 'gpu_id':[0],'device':'cuda','batch_size':1,'batch_size_for_eval':1,
 'nthreads':0,'epoch':EPOCHS,'min_epoch_when_save':0,'save_checkpoint_every':1,
 'save_all_checkpoint':False,'save_dir':SAVE_DIR,'id':RUN_ID,
 'exp_name':f'{RUN_MODE}{len(train)}_e{EPOCHS}_{MASK_MODE}_mask_'+time.strftime('%Y%m%d_%H%M%S'),'start_from':RESUME_RUN_ID,'start_from_mode':'last','pretrain':None,
 'seed':SEED,'random_seed':False,'debug':False,'disable_tqdm':False,
 'train_proposal_sample_num':30,'gt_proposal_sample_num':10,
 'soft_k':20,'window_size':80,'ret_encoder':'multiscale_project',
 'max_caption_len':30,'gpt_beam_size':3,'num_queries':50,'decoder_query_ablation':False,
 'caption_decoder_type':'gpt2','caption_decoder_new':'none',
 'eval_enable_grounding':False,
 'with_box_refine':True,
 # Keep the released DDVC task objectives; add only proposal-mask consistency BCE.
 'bbox_loss_coef':0.0,'set_cost_bbox':0.0,
 'local_visual_attention':MASK_MODE=='hard_local',
 'learned_proposal_mask':MASK_MODE=='learned_gated',
 'proposal_mask_bins':MASK_BINS,'proposal_mask_encoding_dim':MASK_ENCODING_DIM,
 'proposal_mask_loss_coef':MASK_LOSS_COEF,
 'mask_target':MASK_TARGET,'mask_query_conditioning':QUERY_CONDITIONING,
 'mask_prune_threshold':PRUNE_THRESHOLD,'mask_gate_min':GATE_MIN,'mask_gate_max':GATE_MAX,
 'mask_context_ratio':MASK_CONTEXT_RATIO,'mask_min_frames':MASK_MIN_FRAMES,
 'mask_balanced_bce':BALANCED_MASK_BCE,'corrected_query_contrastive':CORRECTED_QUERY_CONTRASTIVE,
 'data_signature':data_signature,
 'local_visual_context_ratio':LOCAL_CONTEXT_RATIO,'local_visual_min_frames':LOCAL_MIN_FRAMES,
 'gpt_layer':2,'add_cross_attention':True,
 # Make the existing reranking defaults explicit for reproducibility.
 'ec_alpha':1.0,'count_loss_coef':0.5,'eval_matching_score_weight':0.0,
 'caption_cost_chunk_size':16
})
(REPO/'cfgs').mkdir(exist_ok=True)
cfg=REPO/'cfgs'/'kaggle_yc2_learned_mask.yml'
cfg.write_text(yaml.safe_dump(opts,sort_keys=False))
print(cfg, '| selected train/val:',len(train),len(val),'| vocab:',len(word_to_ix),'| epochs:',EPOCHS)
print('DDVC mask mode:',MASK_MODE,'| random 2-layer GPT-2 | one GPU')
print('New objective:',{k:opts[k] for k in ['learned_proposal_mask','proposal_mask_loss_coef','proposal_mask_bins']})
print("Reranking:", {k: opts[k] for k in ("ec_alpha", "count_loss_coef", "eval_matching_score_weight")})

# Preserve exactly what was configured and the actual selected data IDs.
run_manifest = {
    'source_commit':SOURCE_COMMIT,'reference_repo':'https://github.com/bhubanendra-mishra/dense-video-cap',
    'reference_commit':'43914e17769701b9cf98eda203ae4c465b315fab',
    'mask_mode':MASK_MODE,'seed':SEED,'epochs':EPOCHS,
    'train_ids':list(train),'val_ids':list(val),
    'mask_target':MASK_TARGET, 'data_signature':data_signature,
    'mask_settings':{k:opts[k] for k in opts if k.startswith('mask_') or k=='corrected_query_contrastive'},
    'mask_placement':'feature gating plus predicted-context attention pruning before GPT-2 cross-attention',
}
manifest_path = REPO/f'run_manifest_{MASK_MODE}.json'
manifest_path.write_text(json.dumps(run_manifest,indent=2),encoding='utf-8')

print('Action-aware mask recipe:',run_manifest['mask_settings'])


## Train and evaluate

Validation uses the selected validation subset in smoke/subset mode and the complete feature-matched validation split in full mode. Training writes an unbuffered log and checkpoints each epoch. Set `RESUME_RUN_ID` in the first cell to the same run ID to continue from `model-last.pth`.


In [ ]:
run_env = os.environ.copy()
run_env.update({'CUDA_VISIBLE_DEVICES':'0','TOKENIZERS_PARALLELISM':'false','PYTHONUNBUFFERED':'1'})
cmd = [sys.executable, '-u', 'train.py', '--cfg_path', str(cfg.relative_to(REPO))]
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
t0 = time.time()
with LOG_PATH.open('a', encoding='utf-8') as log:
    log.write(f"\n\n=== {time.strftime('%Y-%m-%d %H:%M:%S')} | {RUN_MODE} | {len(train)} train / {len(val)} val | {EPOCHS} epochs ===\n")
    log.flush()
    proc = subprocess.Popen(cmd, cwd=REPO, env=run_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    returncode = proc.wait()
if returncode:
    raise subprocess.CalledProcessError(returncode, cmd)
print(f'Completed in {(time.time()-t0)/60:.1f} min; log: {LOG_PATH}')


## Summary / artifacts

In [ ]:
run_root = REPO/SAVE_DIR
runs = sorted([p for p in run_root.iterdir() if p.is_dir()], key=lambda p:p.stat().st_mtime) if run_root.exists() else []
matching_runs = [p for p in runs if p.name.startswith(RUN_ID)]
run = run_root/RESUME_RUN_ID if RESUME_RUN_ID else (matching_runs[-1] if matching_runs else None)
if run is None:
    print('No run directory was created. Check the training log:', LOG_PATH)
else:
    print('Run:', run)
    for name in ['model-best.pth','model-last.pth']:
        print(name + ':', 'present' if (run/name).exists() else 'missing')
    info_path = run/'info.json'
    if info_path.exists():
        info = json.loads(info_path.read_text(encoding='utf-8'))
        best, last = info.get('best',{}), info.get('last',{})
        print('Best epoch:', best.get('epoch'))
        print('Best validation score (METEOR + SODA_c):', best.get('best_val_score'))
        print('Best prediction:', best.get('result_json_path'))
        print('Last evaluated epoch:', last.get('epoch'))
        history = info.get('history',{}).get('val_result_history',{})
        for ep,row in history.items(): print('epoch',ep,row.get('eval_score',{}))
    else:
        print('No info.json yet: training did not complete a validation cycle; see log:', LOG_PATH)
    prediction_dir = run/'prediction'
    final_files = sorted(p for p in prediction_dir.glob('*.json') if '.raw.' not in p.name and not p.name.endswith(('.grounding.json', '.tal.json')))
    print('Final prediction files for visualization:', [str(p) for p in final_files][:20])
    print('Raw candidate files (diagnosis only):', [str(p) for p in prediction_dir.glob('*.raw.json')][:20])
    if info_path.exists() and best.get('result_json_path'):
        best_path = Path(best['result_json_path'])
        if not best_path.is_absolute():
            best_path = REPO/best_path
        raw_path = best_path.with_suffix('.raw.json')
        rerank_path = Path(str(raw_path)+f"_rerank_alpha{opts['ec_alpha']}_temp2.0.json")
        if best_path.exists():
            selected_output = json.loads(best_path.read_text())
            print('Best final JSON (use this for HTML):', best_path)
            print('Selected predictions/video:', selected_output.get('avg_proposal_num'))
            if raw_path.exists():
                print('Raw candidates/video:', json.loads(raw_path.read_text()).get('avg_proposal_num'))
            if rerank_path.exists():
                metric_input = json.loads(rerank_path.read_text())
                assert selected_output['results'] == metric_input['results'], 'Final JSON differs from the reranked metric input.'
                print('PASS: final JSON contains exactly the prediction list used by metrics.')


    # Small result bundle for analysis: predictions, actual references/config and logs.
    import shutil, zipfile
    bundle_dir = Path('/kaggle/working')/f'ddvc_{MASK_MODE}_action_mask_results'
    bundle_dir.mkdir(exist_ok=True)
    for name in ['info.json']:
        if (run/name).exists(): shutil.copy2(run/name, bundle_dir/name)
    for src, name in [(train_json, 'train.json'), (val_json, 'val.json'), (cfg, 'config.yml'), (LOG_PATH, 'train.log')]:
        if src.exists(): shutil.copy2(src, bundle_dir/name)
    if manifest_path.exists(): shutil.copy2(manifest_path,bundle_dir/'run_manifest.json')
    (bundle_dir/'source_commit.txt').write_text(SOURCE_COMMIT+'\n')
    (bundle_dir/'runtime_source.diff').write_text(subprocess.check_output(['git','diff'],cwd=REPO,text=True),encoding='utf-8')
    (bundle_dir/'resolved_packages.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True),encoding='utf-8')
    (bundle_dir/'torch_runtime.json').write_text(json.dumps({'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0)},indent=2))
    archive_path = bundle_dir.with_suffix('.zip')
    with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for item in bundle_dir.iterdir():
            if item.is_file(): archive.write(item, item.name)
        for item in (run/'prediction').glob('*.json'):
            archive.write(item, 'prediction/'+item.name)
    print('Download this result ZIP after training:', archive_path)


## Modes and output

- Default `MASK_MODE='learned_gated'`: learned mask + event-confidence fusion, 30% train/validation, 15 epochs.
- `global`: same-branch baseline without the new mask/objective. `hard_local`: old hard attention-mask control. Use a fresh run for each mode and compare actual selected IDs/configs.
- `RUN_MODE='smoke'`: 1%, 2 epochs. `subset`: 30%, 15 epochs. `full`: all feature-matched data, 15 epochs.
- Run folder: `/kaggle/working/DDVC_action_aware_mask/save_learned_mask/<run-id>/`. Checkpoints: `model-best.pth`, `model-last.pth`.
- Log: `/kaggle/working/ddvc_learned_gated_mask_train.log` in default mode. Inspect `loss_proposal_mask`, `proposal_mask_gate`, `proposal_mask_mean`, `proposal_mask_inside`.
- Result bundle: `/kaggle/working/ddvc_learned_gated_mask_results.zip`. Includes info/history, config, selected GT references, source/run manifest, log and final/raw JSON predictions. Model weights/features are excluded from this small ZIP.
- Use final JSON for HTML/metric inspection. Raw JSON retains all candidates.
- Resume only with the exact saved run folder ID and matching mode/config. To resume after session expiration, save Kaggle output or download/re-attach checkpoints first; temporary working storage is not persistent.
- This ports mask modules from legacy code; it keeps the current released-code DDVC retrieval recipe. It is not an exact reproduction of either the 2018 model or the no-retrieval DDVC paper baseline.


## Resume and interpretation

Output: `/kaggle/working/DDVC_action_aware_mask/save_action_aware_mask/<actual-run-id>/`.
Set `RESUME_RUN_ID` to the exact saved folder name, restore it into this writable location, and retain source, mask settings, epoch budget and data IDs. Format-2 model-last contains its own metric metadata even if info.json was not yet updated. A `validation_pending` checkpoint repeats evaluation before advancing, not its completed training epoch.

Diagnostics in training log: `proposal_mask_gate`, `_gate_min`, `_gate_max`, `_active_fraction`, `_target_iou`. An active fraction near 1 means pruning is weak, not broken. Target IoU measures the thresholded soft-mask target agreement, not detector tIoU. No guaranteed CIDEr/SODA gain is claimed. GT masks only supervise matched training queries; caption generation always uses predictions.
